# Diamond synchrotron HDF tomography -> Dragonfly TIFF stack

Workflow:

`.hdf file -> inspect datasets -> detect raw/reconstructed -> reconstruct/export -> save TIFF stack -> open in Dragonfly`

This notebook is designed for real Diamond Light Source synchrotron tomography HDF5/NeXus files from battery electrode scans. It handles two common cases:

- The HDF file already contains a reconstructed 3D volume.
- The HDF file contains raw projections, flats, darks, and optionally rotation angles.
- The heavy detector data are in `.hdf` / `.h5`, while flats, darks, angles, or `image_key` metadata are in a separate `.nxs` sidecar file.

The output is a folder of individual TIFF slices named like `slice_00000.tif`, `slice_00001.tif`, etc. Dragonfly can import that TIFF image sequence as a 3D stack.

The recommended Dragonfly import target is the `uint16` stack. The `float32` stack is kept as the scientific copy because it preserves reconstructed values more faithfully.

In [1]:
from pathlib import Path

# ---- User-editable paths ----
# Use raw Windows strings: r"C:\path\to\file.hdf"
INPUT_HDF_PATH = Path(r"C:\PhD_programs\hdf_to_tiff\pco1-192241.hdf")

# Optional Diamond NeXus sidecar file. Set this when the `.nxs` contains
# flats, darks, rotation angles, or `image_key` metadata for the HDF detector data.
# Example: NXS_METADATA_PATH = Path(r"C:\path\to\your\scan.nxs")
NXS_METADATA_PATH = Path(r"C:\PhD_programs\hdf_to_tiff\192241.nxs")

OUTPUT_ROOT = Path(r"C:\PhD_programs\hdf_to_tiff\HDF_Output")
PREVIEW_OUTPUT_FOLDER = OUTPUT_ROOT / "preview"
FINAL_TIFF_OUTPUT_FOLDER = OUTPUT_ROOT / "final_tiff_stack"

# ---- User-editable reconstruction settings ----
# Smaller chunks reduce RAM use on Windows. Try 16 or 32 if you hit MemoryError.
CHUNK_SIZE = 64

# Number of detector rows used for the quick preview reconstruction.
PREVIEW_ROW_COUNT = 50

# Full reconstruction can be long and memory-heavy, so it is disabled by default.
RUN_FULL_RECONSTRUCTION = False

# Optional fixed scaling for uint16 output. Leave as None to scale each saved slice
# from percentiles. For comparable intensities across slices, set both values
# after looking at preview statistics, for example UINT16_VMIN = -0.01.
UINT16_VMIN = None
UINT16_VMAX = None
UINT16_LOWER_PERCENTILE = 0.5
UINT16_UPPER_PERCENTILE = 99.5

# ---- Derived output folders ----
PREVIEW_FLOAT32_FOLDER = PREVIEW_OUTPUT_FOLDER / "float32"
PREVIEW_UINT16_FOLDER = PREVIEW_OUTPUT_FOLDER / "uint16"
CENTER_TEST_FOLDER = PREVIEW_OUTPUT_FOLDER / "center_tests_uint16"
FINAL_FLOAT32_FOLDER = FINAL_TIFF_OUTPUT_FOLDER / "float32"
FINAL_UINT16_FOLDER = FINAL_TIFF_OUTPUT_FOLDER / "uint16"

for folder in [
    PREVIEW_OUTPUT_FOLDER,
    PREVIEW_FLOAT32_FOLDER,
    PREVIEW_UINT16_FOLDER,
    CENTER_TEST_FOLDER,
    FINAL_TIFF_OUTPUT_FOLDER,
    FINAL_FLOAT32_FOLDER,
    FINAL_UINT16_FOLDER,
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Input HDF path: {INPUT_HDF_PATH}")
print(f"NXS metadata path: {NXS_METADATA_PATH}")
print(f"Preview output: {PREVIEW_OUTPUT_FOLDER}")
print(f"Final TIFF output: {FINAL_TIFF_OUTPUT_FOLDER}")
print(f"RUN_FULL_RECONSTRUCTION = {RUN_FULL_RECONSTRUCTION}")

Input HDF path: C:\PhD_programs\hdf_to_tiff\pco1-192241.hdf
NXS metadata path: C:\PhD_programs\hdf_to_tiff\192241.nxs
Preview output: C:\PhD_programs\hdf_to_tiff\HDF_Output\preview
Final TIFF output: C:\PhD_programs\hdf_to_tiff\HDF_Output\final_tiff_stack
RUN_FULL_RECONSTRUCTION = False


In [2]:
import gc
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    import h5py
except Exception as exc:
    raise ImportError(
        "h5py is required to inspect Diamond HDF5 files. Install it with "
        "`conda install -c conda-forge h5py` or `pip install h5py`."
    ) from exc

try:
    import tifffile
except Exception as exc:
    raise ImportError(
        "tifffile is required to write TIFF stacks. Install it with "
        "`conda install -c conda-forge tifffile` or `pip install tifffile`."
    ) from exc

try:
    import tomopy

    TOMOPY_AVAILABLE = True
    TOMOPY_IMPORT_ERROR = None
except Exception as exc:
    tomopy = None
    TOMOPY_AVAILABLE = False
    TOMOPY_IMPORT_ERROR = exc

try:
    import dxchange

    DXCHANGE_AVAILABLE = True
except Exception:
    dxchange = None
    DXCHANGE_AVAILABLE = False

print(f"h5py available: {h5py.__version__}")
print(f"numpy available: {np.__version__}")
print(f"tifffile available: {tifffile.__version__}")
print(f"TomoPy available: {TOMOPY_AVAILABLE}")
if not TOMOPY_AVAILABLE:
    print()
    print("TomoPy is not available. You can still inspect HDF files and export already reconstructed volumes.")
    print("Raw projection reconstruction will raise an error until TomoPy is installed.")
    print("On Windows, TomoPy is commonly easiest via WSL2 Ubuntu or conda-forge:")
    print("  conda create -n tomopy310 -c conda-forge python=3.10 tomopy h5py numpy matplotlib tifffile")
    print(f"Original TomoPy import error: {TOMOPY_IMPORT_ERROR}")
print(f"DXchange available: {DXCHANGE_AVAILABLE}")

h5py available: 3.16.0
numpy available: 2.4.2
tifffile available: 2026.3.3
TomoPy available: False

TomoPy is not available. You can still inspect HDF files and export already reconstructed volumes.
Raw projection reconstruction will raise an error until TomoPy is installed.
On Windows, TomoPy is commonly easiest via WSL2 Ubuntu or conda-forge:
  conda create -n tomopy310 -c conda-forge python=3.10 tomopy h5py numpy matplotlib tifffile
Original TomoPy import error: No module named 'tomopy'
DXchange available: False


In [3]:
USEFUL_ATTR_KEYWORDS = (
    "voxel",
    "pixel",
    "spacing",
    "resolution",
    "size",
    "unit",
    "units",
    "scale",
    "calibration",
    "distance",
)


def _decode_hdf_value(value):
    """Convert HDF5 attribute values into readable Python values."""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, np.bytes_):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        if value.size <= 16:
            return [_decode_hdf_value(v) for v in value.tolist()]
        return f"array(shape={value.shape}, dtype={value.dtype})"
    return value


def _attr_is_useful(key):
    key_lower = str(key).lower()
    return any(token in key_lower for token in USEFUL_ATTR_KEYWORDS)


def _format_attrs(attrs):
    if not attrs:
        return []
    lines = []
    for key, value in attrs.items():
        text = str(value)
        if len(text) > 160:
            text = text[:157] + "..."
        lines.append(f"    attr[{key!r}] = {text}")
    return lines


def inspect_hdf_tree(hdf_path, print_attrs=True):
    """
    Recursively inspect an HDF5 file.

    Prints every dataset path, shape, dtype, chunking/compression when present,
    and useful attributes such as voxel size, pixel size, spacing, or units.
    Returns a structured list of dataset metadata dictionaries.
    """
    hdf_path = Path(hdf_path)
    if not hdf_path.exists():
        raise FileNotFoundError(
            f"Input HDF file does not exist: {hdf_path}\n"
            "Edit INPUT_HDF_PATH in the setup cell and run the notebook again."
        )

    metadata = []

    with h5py.File(hdf_path, "r") as h5:
        def visitor(name, obj):
            if not isinstance(obj, h5py.Dataset):
                return

            dataset_path = "/" + name
            attrs = {
                key: _decode_hdf_value(value)
                for key, value in obj.attrs.items()
                if _attr_is_useful(key)
            }
            meta = {
                "path": dataset_path,
                "shape": tuple(obj.shape),
                "ndim": obj.ndim,
                "dtype": str(obj.dtype),
                "attrs": attrs,
                "chunks": obj.chunks,
                "compression": obj.compression,
            }
            metadata.append(meta)

            extra = []
            if obj.chunks is not None:
                extra.append(f"chunks={obj.chunks}")
            if obj.compression is not None:
                extra.append(f"compression={obj.compression}")
            extra_text = " | " + " | ".join(extra) if extra else ""
            print(f"{dataset_path} | shape={obj.shape} | dtype={obj.dtype}{extra_text}")
            if print_attrs:
                for line in _format_attrs(attrs):
                    print(line)

        h5.visititems(visitor)

    print()
    print(f"Found {len(metadata)} datasets in {hdf_path.name}.")
    return metadata

In [4]:
PROJECTION_TOKENS = ("data", "proj", "projection", "tomo")
FLAT_TOKENS = ("flat", "white", "bright")
DARK_TOKENS = ("dark", "black")
ANGLE_TOKENS = ("theta", "angle", "rotation")
RECON_TOKENS = ("recon", "reconstruction", "volume", "slice", "result", "processed")
IMAGE_KEY_TOKENS = ("image_key", "imagekey", "image key")


def _contains_any(text, tokens):
    text = str(text).lower()
    return any(token in text for token in tokens)


def _candidate_sort_key(meta):
    # Prefer larger datasets, then shorter paths.
    shape = meta.get("shape", ())
    n_items = 1
    for dim in shape:
        n_items *= max(int(dim), 1)
    return (-n_items, len(meta.get("path", "")), meta.get("path", ""))


def detect_candidates(metadata):
    """
    Detect likely projection, flat, dark, theta, and reconstructed-volume datasets.
    """
    candidates = {
        "projection": [],
        "flat": [],
        "dark": [],
        "angle": [],
        "reconstructed_volume": [],
        "image_key": [],
    }

    for meta in metadata:
        path_lower = meta["path"].lower()
        ndim = meta["ndim"]

        is_flat = _contains_any(path_lower, FLAT_TOKENS)
        is_dark = _contains_any(path_lower, DARK_TOKENS)
        is_recon = _contains_any(path_lower, RECON_TOKENS)

        if ndim == 3 and _contains_any(path_lower, PROJECTION_TOKENS) and not (is_flat or is_dark or is_recon):
            candidates["projection"].append(meta)

        if is_flat and ndim in (2, 3):
            candidates["flat"].append(meta)

        if is_dark and ndim in (2, 3):
            candidates["dark"].append(meta)

        if ndim == 1 and _contains_any(path_lower, ANGLE_TOKENS):
            candidates["angle"].append(meta)

        if ndim == 3 and is_recon:
            candidates["reconstructed_volume"].append(meta)

        if ndim == 1 and _contains_any(path_lower, IMAGE_KEY_TOKENS):
            candidates["image_key"].append(meta)

    for key in candidates:
        candidates[key] = sorted(candidates[key], key=_candidate_sort_key)

    return candidates


def print_candidates(candidates):
    """Print candidate datasets in a readable indexed list."""
    for kind, metas in candidates.items():
        print()
        print(f"{kind} candidates ({len(metas)}):")
        if not metas:
            print("  none detected")
            continue
        for index, meta in enumerate(metas):
            attrs = meta.get("attrs", {})
            attr_text = ""
            if attrs:
                attr_text = " | attrs: " + ", ".join(f"{k}={v}" for k, v in attrs.items())
            print(
                f"  [{index}] {meta['path']} | shape={meta['shape']} | "
                f"dtype={meta['dtype']}{attr_text}"
            )

In [5]:
# Run this cell after editing INPUT_HDF_PATH and, if present, NXS_METADATA_PATH.
dataset_metadata_by_source = {}
candidates_by_source = {}

print("=== Input HDF detector file ===")
dataset_metadata_by_source["input_hdf"] = inspect_hdf_tree(INPUT_HDF_PATH)
candidates_by_source["input_hdf"] = detect_candidates(dataset_metadata_by_source["input_hdf"])
print_candidates(candidates_by_source["input_hdf"])

if NXS_METADATA_PATH is not None:
    print()
    print("=== Optional NXS metadata/reference file ===")
    dataset_metadata_by_source["nxs"] = inspect_hdf_tree(NXS_METADATA_PATH)
    candidates_by_source["nxs"] = detect_candidates(dataset_metadata_by_source["nxs"])
    print_candidates(candidates_by_source["nxs"])
else:
    print()
    print("NXS_METADATA_PATH is None, so only the input HDF file was inspected.")
    print("If flats/darks/theta are in a .nxs file, set NXS_METADATA_PATH in the setup cell and rerun this cell.")

# Backward-compatible aliases for the main input file.
dataset_metadata = dataset_metadata_by_source["input_hdf"]
candidates = candidates_by_source["input_hdf"]

=== Input HDF detector file ===
/entry/data/data | shape=(2581, 2160, 2560) | dtype=uint16 | chunks=(1, 2160, 2560)

Found 1 datasets in pco1-192241.hdf.

projection candidates (1):
  [0] /entry/data/data | shape=(2581, 2160, 2560) | dtype=uint16

flat candidates (0):
  none detected

dark candidates (0):
  none detected

angle candidates (0):
  none detected

reconstructed_volume candidates (0):
  none detected

image_key candidates (0):
  none detected

=== Optional NXS metadata/reference file ===
/entry1/before_scan/cs1/cs1_x | shape=() | dtype=float64
/entry1/before_scan/cs1/cs1_y | shape=() | dtype=float64
/entry1/before_scan/cs1/cs1_z | shape=() | dtype=float64
/entry1/before_scan/det_cfg/adc_mode | shape=() | dtype=object
/entry1/before_scan/det_cfg/cam_bin_x | shape=() | dtype=object
/entry1/before_scan/det_cfg/cam_bin_y | shape=() | dtype=object
/entry1/before_scan/det_cfg/cam_img_size_x | shape=() | dtype=object
/entry1/before_scan/det_cfg/cam_img_size_y | shape=() | dtype=ob

In [6]:
# ---- User-editable HDF/NXS source files and dataset paths ----
# After running candidate detection, replace these with paths from the printed candidate lists.
#
# Typical Diamond split:
# - projections: main .hdf / .h5 detector data file
# - flats/darks/theta/image_key: .nxs NeXus scan file

proj_file = INPUT_HDF_PATH

# If the .nxs file holds flats/darks/theta metadata, keep these pointed at NXS_METADATA_PATH.
# If not, set them to INPUT_HDF_PATH.
flat_file = NXS_METADATA_PATH if NXS_METADATA_PATH is not None else INPUT_HDF_PATH
dark_file = NXS_METADATA_PATH if NXS_METADATA_PATH is not None else INPUT_HDF_PATH
theta_file = NXS_METADATA_PATH if NXS_METADATA_PATH is not None else INPUT_HDF_PATH
image_key_file = NXS_METADATA_PATH if NXS_METADATA_PATH is not None else INPUT_HDF_PATH

# The dataset that contains the raw projections. Your detected candidate was likely /entry/data/data.
proj_path = "/entry/data/data"

# Direct flat/dark datasets, if present. Leave as None to try the NXtomo image_key fallback below.
flat_path = None
dark_path = None

# Common NeXus angle paths. Edit after inspecting the .nxs candidates.
theta_path = "/entry/data/rotation_angle"

# Optional NXtomo-style image_key support.
# image_key convention is normally: 0 = projection, 1 = flat/white, 2 = dark/black.
image_key_path = "/entry/instrument/detector/image_key"
image_key_data_file = NXS_METADATA_PATH if NXS_METADATA_PATH is not None else INPUT_HDF_PATH
image_key_data_path = "/entry/data/data"
FLAT_IMAGE_KEY_VALUES = (1,)
DARK_IMAGE_KEY_VALUES = (2,)
PROJECTION_IMAGE_KEY_VALUES = (0,)

# If the file already contains a reconstructed volume, set recon_path and recon_file.
# If recon_path is not None, the reconstructed-volume export cell can be used without TomoPy.
recon_file = INPUT_HDF_PATH
recon_path = None

print("Selected files and paths:")
print(f"  proj_file           = {proj_file}")
print(f"  proj_path           = {proj_path}")
print(f"  flat_file           = {flat_file}")
print(f"  flat_path           = {flat_path}")
print(f"  dark_file           = {dark_file}")
print(f"  dark_path           = {dark_path}")
print(f"  theta_file          = {theta_file}")
print(f"  theta_path          = {theta_path}")
print(f"  image_key_file      = {image_key_file}")
print(f"  image_key_path      = {image_key_path}")
print(f"  image_key_data_file = {image_key_data_file}")
print(f"  image_key_data_path = {image_key_data_path}")
print(f"  recon_file          = {recon_file}")
print(f"  recon_path          = {recon_path}")

Selected files and paths:
  proj_file           = C:\PhD_programs\hdf_to_tiff\pco1-192241.hdf
  proj_path           = /entry/data/data
  flat_file           = C:\PhD_programs\hdf_to_tiff\192241.nxs
  flat_path           = None
  dark_file           = C:\PhD_programs\hdf_to_tiff\192241.nxs
  dark_path           = None
  theta_file          = C:\PhD_programs\hdf_to_tiff\192241.nxs
  theta_path          = /entry/data/rotation_angle
  image_key_file      = C:\PhD_programs\hdf_to_tiff\192241.nxs
  image_key_path      = /entry/instrument/detector/image_key
  image_key_data_file = C:\PhD_programs\hdf_to_tiff\192241.nxs
  image_key_data_path = /entry/data/data
  recon_file          = C:\PhD_programs\hdf_to_tiff\pco1-192241.hdf
  recon_path          = None


In [ ]:
ROLE_TO_CANDIDATE_KEY = {
    "projection": "projection",
    "proj": "projection",
    "flat": "flat",
    "dark": "dark",
    "theta": "angle",
    "angle": "angle",
    "recon": "reconstructed_volume",
    "reconstruction": "reconstructed_volume",
    "reconstructed_volume": "reconstructed_volume",
    "image_key": "image_key",
}


def _normalise_hdf_path(dataset_path):
    if dataset_path is None:
        return None
    dataset_path = str(dataset_path).strip()
    if not dataset_path:
        return None
    if not dataset_path.startswith("/"):
        dataset_path = "/" + dataset_path
    return dataset_path


def _print_candidate_paths(role=None):
    if "candidates_by_source" not in globals() and "candidates" not in globals():
        print("Candidate lists are not available yet. Run the HDF inspection and candidate detection cells first.")
        return

    if role is None:
        if "candidates_by_source" in globals():
            keys = sorted({key for source_candidates in candidates_by_source.values() for key in source_candidates})
        else:
            keys = list(candidates.keys())
    else:
        keys = [ROLE_TO_CANDIDATE_KEY.get(role, role)]

    if "candidates_by_source" in globals():
        for source_name, source_candidates in candidates_by_source.items():
            print(f"Source: {source_name}")
            for key in keys:
                print(f"  {key} candidates:")
                metas = source_candidates.get(key, [])
                if not metas:
                    print("    none detected")
                for meta in metas:
                    print(f"    {meta['path']} | shape={meta['shape']} | dtype={meta['dtype']}")
    else:
        for key in keys:
            print(f"{key} candidates:")
            metas = candidates.get(key, [])
            if not metas:
                print("  none detected")
            for meta in metas:
                print(f"  {meta['path']} | shape={meta['shape']} | dtype={meta['dtype']}")


def _normalise_file_path(file_path, default=None):
    if file_path is None:
        file_path = default
    if file_path is None:
        return None
    return Path(file_path)


def hdf_path_exists(hdf_path, dataset_path):
    """Return True if dataset_path exists inside hdf_path."""
    dataset_path = _normalise_hdf_path(dataset_path)
    if dataset_path is None:
        return False
    hdf_path = Path(hdf_path)
    if not hdf_path.exists():
        return False
    with h5py.File(hdf_path, "r") as h5:
        return dataset_path in h5


def _require_dataset(h5, dataset_path, role):
    dataset_path = _normalise_hdf_path(dataset_path)
    if dataset_path is None or dataset_path not in h5:
        print(f"Dataset path for {role!r} was not found: {dataset_path}")
        _print_candidate_paths(role)
        raise KeyError(
            f"Invalid HDF dataset path for {role}: {dataset_path}. "
            "Use one of the candidate paths printed above or inspect the HDF tree."
        )
    obj = h5[dataset_path]
    if not isinstance(obj, h5py.Dataset):
        raise TypeError(f"Path exists but is not a dataset: {dataset_path}")
    return obj, dataset_path


def print_dataset_info(hdf_path, dataset_path):
    """Print shape, dtype, chunking, compression, and useful attributes for one dataset."""
    dataset_path = _normalise_hdf_path(dataset_path)
    if dataset_path is None:
        print("No dataset path was supplied.")
        return
    with h5py.File(hdf_path, "r") as h5:
        if dataset_path not in h5:
            print(f"Dataset does not exist: {dataset_path}")
            _print_candidate_paths()
            return
        obj = h5[dataset_path]
        if not isinstance(obj, h5py.Dataset):
            print(f"Path exists but is not a dataset: {dataset_path}")
            return

        print(f"{dataset_path}")
        print(f"  shape       : {obj.shape}")
        print(f"  dtype       : {obj.dtype}")
        print(f"  ndim        : {obj.ndim}")
        print(f"  chunks      : {obj.chunks}")
        print(f"  compression : {obj.compression}")
        useful_attrs = {
            key: _decode_hdf_value(value)
            for key, value in obj.attrs.items()
            if _attr_is_useful(key)
        }
        if useful_attrs:
            print("  useful attrs:")
            for key, value in useful_attrs.items():
                print(f"    {key}: {value}")


def normalise_to_uint16(image, vmin=None, vmax=None, lower_percentile=None, upper_percentile=None):
    """
    Convert an image to uint16 for Dragonfly-friendly TIFF export.

    If vmin/vmax are not supplied, robust percentiles are computed from finite pixels.
    For full-stack quantitative comparison, set fixed UINT16_VMIN and UINT16_VMAX.
    """
    arr = np.asarray(image, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros(arr.shape, dtype=np.uint16)

    if lower_percentile is None:
        lower_percentile = globals().get("UINT16_LOWER_PERCENTILE", 0.5)
    if upper_percentile is None:
        upper_percentile = globals().get("UINT16_UPPER_PERCENTILE", 99.5)

    if vmin is None:
        vmin = float(np.percentile(finite, lower_percentile))
    if vmax is None:
        vmax = float(np.percentile(finite, upper_percentile))

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin = float(np.nanmin(finite))
        vmax = float(np.nanmax(finite))
    if vmax <= vmin:
        return np.zeros(arr.shape, dtype=np.uint16)

    scaled = (arr - vmin) / (vmax - vmin)
    scaled = np.nan_to_num(scaled, nan=0.0, posinf=1.0, neginf=0.0)
    scaled = np.clip(scaled, 0.0, 1.0)
    return np.round(scaled * 65535.0).astype(np.uint16)


def save_tiff_stack_float32(volume, output_folder, start_index=0, prefix="slice", progress_every=100):
    """Save a 3D NumPy array as one float32 TIFF per z slice."""
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(volume)
    if arr.ndim == 2:
        arr = arr[np.newaxis, :, :]
    if arr.ndim != 3:
        raise ValueError(f"Expected a 3D volume or 2D image, got shape {arr.shape}")

    for local_index in range(arr.shape[0]):
        out_index = start_index + local_index
        out_path = output_folder / f"{prefix}_{out_index:05d}.tif"
        tifffile.imwrite(out_path, np.asarray(arr[local_index], dtype=np.float32), photometric="minisblack")
        if local_index % progress_every == 0:
            print(f"  float32 saved {out_path.name}")


def save_tiff_stack_uint16(
    volume,
    output_folder,
    start_index=0,
    prefix="slice",
    vmin=None,
    vmax=None,
    progress_every=100,
):
    """Save a 3D NumPy array as one uint16 TIFF per z slice."""
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(volume)
    if arr.ndim == 2:
        arr = arr[np.newaxis, :, :]
    if arr.ndim != 3:
        raise ValueError(f"Expected a 3D volume or 2D image, got shape {arr.shape}")

    for local_index in range(arr.shape[0]):
        out_index = start_index + local_index
        out_path = output_folder / f"{prefix}_{out_index:05d}.tif"
        image_u16 = normalise_to_uint16(arr[local_index], vmin=vmin, vmax=vmax)
        tifffile.imwrite(out_path, image_u16, photometric="minisblack")
        if local_index % progress_every == 0:
            print(f"  uint16 saved {out_path.name}")


def show_middle_slice(volume, title="Middle slice", cmap="gray"):
    """Display the middle z slice of a 3D volume, or a 2D image directly."""
    arr = np.asarray(volume)
    if arr.ndim == 3:
        image = arr[arr.shape[0] // 2]
        subtitle = f"{title} | z={arr.shape[0] // 2}"
    elif arr.ndim == 2:
        image = arr
        subtitle = title
    else:
        raise ValueError(f"Expected 2D or 3D data, got shape {arr.shape}")

    finite = image[np.isfinite(image)]
    if finite.size:
        vmin, vmax = np.percentile(finite, [0.5, 99.5])
    else:
        vmin, vmax = None, None

    plt.figure(figsize=(7, 7))
    plt.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.title(subtitle)
    plt.axis("off")
    plt.colorbar(shrink=0.75)
    plt.show()


def _read_dataset_slab(ds, row_start, row_stop, detector_rows, detector_cols, role, frame_indices=None):
    """
    Read a detector-row slab from a 2D/3D flat, dark, or data dataset.

    Supported shapes:
    - (n_frames, detector_rows, detector_cols)
    - (detector_rows, detector_cols)
    - (detector_rows, detector_cols, n_frames), converted to frames-first
    """
    if ds.ndim == 3 and ds.shape[1] == detector_rows and ds.shape[2] == detector_cols:
        if frame_indices is None:
            slab = ds[:, row_start:row_stop, :]
        else:
            slab = ds[frame_indices, row_start:row_stop, :]
    elif ds.ndim == 3 and ds.shape[0] == detector_rows and ds.shape[1] == detector_cols:
        if frame_indices is None:
            slab = np.moveaxis(ds[row_start:row_stop, :, :], -1, 0)
        else:
            slab = np.moveaxis(ds[row_start:row_stop, :, frame_indices], -1, 0)
    elif frame_indices is None and ds.ndim == 2 and ds.shape[0] == detector_rows and ds.shape[1] == detector_cols:
        slab = ds[row_start:row_stop, :]
    else:
        raise ValueError(
            f"{role} dataset shape {ds.shape} is not compatible with projection detector shape "
            f"({detector_rows}, {detector_cols}). Expected either "
            f"(n_frames, {detector_rows}, {detector_cols}), "
            f"({detector_rows}, {detector_cols}), or "
            f"({detector_rows}, {detector_cols}, n_frames)."
        )

    return np.asarray(slab, dtype=np.float32)


def _load_reference_slab_from_file(ref_file, ref_path, row_start, row_stop, detector_rows, detector_cols, role):
    """Load direct flat/dark dataset from either the HDF detector file or a .nxs file."""
    ref_file = _normalise_file_path(ref_file, default=INPUT_HDF_PATH)
    if ref_file is None:
        raise ValueError(f"No file was supplied for {role}.")

    with h5py.File(ref_file, "r") as h5:
        ref_ds, ref_path = _require_dataset(h5, ref_path, role)
        slab = _read_dataset_slab(ref_ds, row_start, row_stop, detector_rows, detector_cols, role)

    print(f"Loaded {role} from {ref_file}::{ref_path}")
    return slab


def _load_image_key_values(image_key_file, image_key_path):
    image_key_file = _normalise_file_path(image_key_file)
    image_key_path = _normalise_hdf_path(image_key_path)
    if image_key_file is None or image_key_path is None:
        return None
    with h5py.File(image_key_file, "r") as h5:
        if image_key_path not in h5:
            print(f"image_key dataset was not found: {image_key_file}::{image_key_path}")
            _print_candidate_paths("image_key")
            return None
        image_key = np.asarray(h5[image_key_path][...]).reshape(-1)
    return image_key


def describe_image_key(image_key_file, image_key_path):
    """Print counts for an NXtomo image_key dataset."""
    image_key = _load_image_key_values(image_key_file, image_key_path)
    if image_key is None:
        return None
    values, counts = np.unique(image_key, return_counts=True)
    print(f"image_key counts from {image_key_file}::{image_key_path}")
    for value, count in zip(values, counts):
        meaning = {0: "projection", 1: "flat/white", 2: "dark/black"}.get(int(value), "other")
        print(f"  {value}: {count} ({meaning})")
    return image_key


def _load_reference_slab_from_image_key(
    data_file,
    data_path,
    image_key_file,
    image_key_path,
    row_start,
    row_stop,
    detector_rows,
    detector_cols,
    role,
    key_values,
):
    """
    Load flats/darks from a 3D data dataset using an NXtomo image_key array.

    This is useful when the .nxs file stores or links the full acquisition sequence
    and `image_key` marks each frame as projection, flat, or dark.
    """
    data_file = _normalise_file_path(data_file, default=INPUT_HDF_PATH)
    data_path = _normalise_hdf_path(data_path)
    image_key = _load_image_key_values(image_key_file, image_key_path)
    if image_key is None:
        return None

    with h5py.File(data_file, "r") as h5:
        data_ds, data_path = _require_dataset(h5, data_path, "image_key_data")
        if data_ds.ndim != 3:
            raise ValueError(f"image_key_data dataset {data_path} must be 3D, got shape {data_ds.shape}")
        if image_key.size != data_ds.shape[0]:
            raise ValueError(
                f"image_key has {image_key.size} entries, but {data_file}::{data_path} "
                f"has {data_ds.shape[0]} frames. These must match to derive {role} frames."
            )

        key_values = np.asarray(tuple(key_values))
        frame_indices = np.flatnonzero(np.isin(image_key, key_values))
        if frame_indices.size == 0:
            raise ValueError(
                f"No {role} frames found in image_key for values {tuple(key_values)}. "
                "NXtomo convention is usually 0=projection, 1=flat, 2=dark; edit "
                "FLAT_IMAGE_KEY_VALUES or DARK_IMAGE_KEY_VALUES if your scan uses different labels."
            )

        slab = _read_dataset_slab(
            data_ds,
            row_start,
            row_stop,
            detector_rows,
            detector_cols,
            role,
            frame_indices=frame_indices,
        )

    print(
        f"Loaded {role} from image_key values {tuple(key_values)} using "
        f"{data_file}::{data_path}; frames={frame_indices.size}"
    )
    return slab


def _load_reference_data(
    role,
    direct_file,
    direct_path,
    image_key_data_file,
    image_key_data_path,
    image_key_file,
    image_key_path,
    image_key_values,
    row_start,
    row_stop,
    detector_rows,
    detector_cols,
):
    """Load flat/dark data either from a direct dataset or via image_key fallback."""
    if _normalise_hdf_path(direct_path) is not None:
        return _load_reference_slab_from_file(
            direct_file,
            direct_path,
            row_start,
            row_stop,
            detector_rows,
            detector_cols,
            role,
        )

    try:
        image_key_slab = _load_reference_slab_from_image_key(
            image_key_data_file,
            image_key_data_path,
            image_key_file,
            image_key_path,
            row_start,
            row_stop,
            detector_rows,
            detector_cols,
            role,
            image_key_values,
        )
        if image_key_slab is not None:
            return image_key_slab
    except Exception as exc:
        print(f"Could not derive {role} frames from image_key: {exc}")

    _print_candidate_paths(role)
    _print_candidate_paths("image_key")
    raise KeyError(
        f"No usable {role} data were found. Set {role}_path to a direct dataset, or set "
        "image_key_file/image_key_path plus image_key_data_file/image_key_data_path so the notebook "
        "can extract flats/darks from an NXtomo acquisition sequence."
    )


def _convert_theta_to_radians(theta):
    theta = np.asarray(theta, dtype=np.float32).reshape(-1)
    finite_theta = theta[np.isfinite(theta)]
    if finite_theta.size and np.nanmax(np.abs(finite_theta)) > (2 * np.pi + 0.1):
        print("Theta values appear to be in degrees; converting to radians for TomoPy.")
        theta = np.deg2rad(theta).astype(np.float32)
    return theta.astype(np.float32)


def _load_theta_from_file(
    theta_file,
    theta_path,
    n_angles,
    image_key_file=None,
    image_key_path=None,
    projection_key_values=(0,),
):
    """
    Load theta in radians from HDF/NXS, optionally filtering by image_key.

    If theta is missing, an evenly spaced 0..pi array is generated with a clear warning.
    """
    theta_file = _normalise_file_path(theta_file, default=INPUT_HDF_PATH)
    theta_path = _normalise_hdf_path(theta_path)

    if theta_file is not None and theta_path is not None:
        with h5py.File(theta_file, "r") as h5:
            if theta_path in h5:
                theta = np.asarray(h5[theta_path][...], dtype=np.float32).reshape(-1)
                if theta.size == n_angles:
                    return _convert_theta_to_radians(theta)

                image_key = _load_image_key_values(image_key_file, image_key_path)
                if image_key is not None and image_key.size == theta.size:
                    projection_mask = np.isin(image_key, tuple(projection_key_values))
                    theta_projection = theta[projection_mask]
                    if theta_projection.size == n_angles:
                        print(
                            f"Theta has {theta.size} values; filtered to {theta_projection.size} "
                            "projection angles using image_key."
                        )
                        return _convert_theta_to_radians(theta_projection)

                raise ValueError(
                    f"Theta dataset {theta_file}::{theta_path} has {theta.size} values, but projection "
                    f"data has {n_angles} angles. Expected one theta per projection, or an image_key "
                    "that can filter theta down to projection frames."
                )

            print(f"Theta dataset was not found: {theta_file}::{theta_path}")
            _print_candidate_paths("theta")

    warnings.warn(
        "No theta dataset was found or theta_path is None. Generating evenly spaced angles "
        "from 0 to pi radians. Verify this is correct for your scan before full reconstruction.",
        RuntimeWarning,
    )
    return np.linspace(0.0, np.pi, n_angles, dtype=np.float32)


def load_projection_slab(
    proj_file,
    proj_path,
    row_start,
    row_stop,
    flat_path=None,
    dark_path=None,
    theta_path=None,
    flat_file=None,
    dark_file=None,
    theta_file=None,
    image_key_file=None,
    image_key_path=None,
    image_key_data_file=None,
    image_key_data_path=None,
    flat_image_key_values=(1,),
    dark_image_key_values=(2,),
    projection_image_key_values=(0,),
):
    """
    Load a slab of raw projection data and matching flats/darks/theta.

    Projection data normally have shape (n_angles, detector_rows, detector_cols).
    Flats/darks/theta may come from the same HDF file or from a separate .nxs file.
    """
    proj_file = _normalise_file_path(proj_file, default=INPUT_HDF_PATH)
    flat_file = _normalise_file_path(flat_file, default=proj_file)
    dark_file = _normalise_file_path(dark_file, default=proj_file)
    theta_file = _normalise_file_path(theta_file, default=proj_file)
    image_key_data_file = _normalise_file_path(image_key_data_file, default=proj_file)
    if image_key_data_path is None:
        image_key_data_path = proj_path

    with h5py.File(proj_file, "r") as h5:
        proj_ds, proj_path = _require_dataset(h5, proj_path, "projection")
        if proj_ds.ndim != 3:
            raise ValueError(
                f"Projection dataset {proj_file}::{proj_path} has shape {proj_ds.shape}; expected "
                "(n_angles, detector_rows, detector_cols)."
            )

        n_angles, detector_rows, detector_cols = proj_ds.shape
        row_start = int(max(0, row_start))
        row_stop = int(min(detector_rows, row_stop))
        if row_stop <= row_start:
            raise ValueError(f"Invalid detector-row slab: row_start={row_start}, row_stop={row_stop}")

        proj = np.asarray(proj_ds[:, row_start:row_stop, :], dtype=np.float32)

    flat = _load_reference_data(
        "flat",
        flat_file,
        flat_path,
        image_key_data_file,
        image_key_data_path,
        image_key_file,
        image_key_path,
        flat_image_key_values,
        row_start,
        row_stop,
        detector_rows,
        detector_cols,
    )
    dark = _load_reference_data(
        "dark",
        dark_file,
        dark_path,
        image_key_data_file,
        image_key_data_path,
        image_key_file,
        image_key_path,
        dark_image_key_values,
        row_start,
        row_stop,
        detector_rows,
        detector_cols,
    )
    theta = _load_theta_from_file(
        theta_file,
        theta_path,
        n_angles,
        image_key_file=image_key_file,
        image_key_path=image_key_path,
        projection_key_values=projection_image_key_values,
    )

    print(f"Loaded projection slab rows [{row_start}:{row_stop}] from {proj_file}::{proj_path}")
    print(f"  proj shape : {proj.shape}")
    print(f"  flat shape : {flat.shape}")
    print(f"  dark shape : {dark.shape}")
    print(f"  theta shape: {theta.shape}")
    return proj, flat, dark, theta


def _tomopy_missing_message():
    return (
        "TomoPy is required for raw projection reconstruction, but it is not available in this Python environment.\n"
        "You can still inspect HDF files and export already reconstructed volumes.\n"
        "Recommended install route on Windows: use WSL2 Ubuntu or conda-forge, for example:\n"
        "  conda create -n tomopy310 -c conda-forge python=3.10 tomopy h5py numpy matplotlib tifffile"
    )


def reconstruct_slab(
    proj,
    flat,
    dark,
    theta,
    center=None,
    apply_stripe_removal=True,
    stripe_sigma=2,
    stripe_level=5,
    algorithm="gridrec",
    mask_ratio=0.95,
):
    """
    Reconstruct a projection slab with TomoPy.

    Steps:
    1. tomopy.normalize(proj, flat, dark)
    2. clip to avoid log issues
    3. tomopy.minus_log
    4. optional tomopy.remove_stripe_fw
    5. tomopy.find_center if center is None
    6. tomopy.recon(..., algorithm="gridrec")
    7. tomopy.circ_mask(..., axis=0, ratio=0.95)
    """
    if not TOMOPY_AVAILABLE:
        raise ImportError(_tomopy_missing_message()) from TOMOPY_IMPORT_ERROR

    proj = np.asarray(proj, dtype=np.float32)
    flat = np.asarray(flat, dtype=np.float32)
    dark = np.asarray(dark, dtype=np.float32)
    theta = np.asarray(theta, dtype=np.float32)

    if proj.ndim != 3:
        raise ValueError(f"Expected proj shape (n_angles, rows, cols), got {proj.shape}")
    if theta.ndim != 1 or theta.size != proj.shape[0]:
        raise ValueError(f"Expected theta shape ({proj.shape[0]},), got {theta.shape}")

    print("Normalising projections with flats and darks...")
    norm = tomopy.normalize(proj, flat, dark)
    norm = np.asarray(norm, dtype=np.float32)
    np.nan_to_num(norm, copy=False, nan=1.0, posinf=1.0, neginf=1.0e-6)
    np.maximum(norm, 1.0e-6, out=norm)

    print("Applying minus-log transform...")
    data = tomopy.minus_log(norm)
    data = np.asarray(data, dtype=np.float32)
    np.nan_to_num(data, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    if apply_stripe_removal:
        print("Applying ring/stripe correction with tomopy.remove_stripe_fw...")
        try:
            data = tomopy.remove_stripe_fw(data, sigma=stripe_sigma, level=stripe_level, pad=True)
        except Exception as exc:
            warnings.warn(
                f"tomopy.remove_stripe_fw failed ({exc}). Continuing without stripe correction.",
                RuntimeWarning,
            )

    if center is None:
        center_index = data.shape[1] // 2
        initial_center = data.shape[2] / 2.0
        print(f"Estimating centre of rotation near detector column {initial_center:.2f}...")
        try:
            center = tomopy.find_center(
                data,
                theta,
                ind=center_index,
                init=initial_center,
                tol=0.5,
                mask=True,
                ratio=1.0,
            )
        except TypeError:
            center = tomopy.find_center(data, theta, ind=center_index, init=initial_center, tol=0.5)
        except Exception as exc:
            warnings.warn(
                f"tomopy.find_center failed ({exc}). Falling back to detector midpoint {initial_center:.2f}.",
                RuntimeWarning,
            )
            center = initial_center

    center = float(center)
    print(f"Reconstructing with centre={center:.3f}, algorithm={algorithm!r}...")
    recon = tomopy.recon(data, theta, center=center, algorithm=algorithm)
    recon = tomopy.circ_mask(recon, axis=0, ratio=mask_ratio)
    return np.asarray(recon, dtype=np.float32), center

In [ ]:
# Check selected files and paths before reconstruction/export.
for role, selected_file, selected_path in [
    ("projection", proj_file, proj_path),
    ("flat", flat_file, flat_path),
    ("dark", dark_file, dark_path),
    ("theta", theta_file, theta_path),
    ("image_key", image_key_file, image_key_path),
    ("image_key_data", image_key_data_file, image_key_data_path),
    ("recon", recon_file, recon_path),
]:
    print()
    print(f"--- {role} ---")
    if selected_path is None:
        print("not set")
    elif selected_file is None:
        print(f"path set but file is None: {selected_path}")
    else:
        print(f"file: {selected_file}")
        print_dataset_info(selected_file, selected_path)

if image_key_path is not None and image_key_file is not None:
    print()
    describe_image_key(image_key_file, image_key_path)

In [ ]:
def collect_voxel_size_attributes(h5, dataset_path):
    """
    Collect useful voxel/pixel/unit attributes from the dataset and its parent groups.
    Diamond/Nexus files often store metadata on parent groups rather than the data dataset itself.
    """
    dataset_path = _normalise_hdf_path(dataset_path)
    parts = [part for part in dataset_path.strip("/").split("/") if part]

    paths_to_check = ["/"]
    current = ""
    for part in parts:
        current += "/" + part
        paths_to_check.append(current)

    records = []
    for path in paths_to_check:
        if path not in h5:
            continue
        obj = h5[path]
        for key, value in obj.attrs.items():
            if _attr_is_useful(key):
                records.append((path, key, _decode_hdf_value(value)))
    return records


def write_metadata_text(output_folder, filename, lines):
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    metadata_path = output_folder / filename
    metadata_path.write_text("\n".join(str(line) for line in lines) + "\n", encoding="utf-8")
    print(f"Wrote metadata: {metadata_path}")
    return metadata_path


def export_hdf_recon_dataset_to_tiff_stacks(
    hdf_path,
    recon_path,
    float32_folder,
    uint16_folder,
    uint16_vmin=None,
    uint16_vmax=None,
):
    """
    Export an already reconstructed 3D HDF dataset as float32 and uint16 TIFF stacks.

    The dataset is read one slice at a time, so the whole volume is not loaded into memory.
    Assumes reconstructed volume shape is (z, y, x).
    """
    hdf_path = Path(hdf_path)
    recon_path = _normalise_hdf_path(recon_path)
    float32_folder = Path(float32_folder)
    uint16_folder = Path(uint16_folder)
    float32_folder.mkdir(parents=True, exist_ok=True)
    uint16_folder.mkdir(parents=True, exist_ok=True)

    with h5py.File(hdf_path, "r") as h5:
        recon_ds, recon_path = _require_dataset(h5, recon_path, "reconstructed_volume")
        if recon_ds.ndim != 3:
            raise ValueError(
                f"Reconstructed volume dataset {recon_path} has shape {recon_ds.shape}; "
                "expected a 3D dataset shaped like (z, y, x)."
            )

        shape = tuple(recon_ds.shape)
        dtype = str(recon_ds.dtype)
        voxel_attrs = collect_voxel_size_attributes(h5, recon_path)

        metadata_lines = [
            "Diamond HDF reconstructed-volume TIFF export",
            f"source_file: {hdf_path}",
            f"dataset_path: {recon_path}",
            f"dataset_shape: {shape}",
            f"dataset_dtype: {dtype}",
            "",
            "voxel_or_pixel_size_attributes_found:",
        ]
        if voxel_attrs:
            for path, key, value in voxel_attrs:
                metadata_lines.append(f"{path} :: {key} = {value}")
        else:
            metadata_lines.append("none found")

        write_metadata_text(float32_folder, "source_metadata.txt", metadata_lines)
        write_metadata_text(uint16_folder, "source_metadata.txt", metadata_lines)

        print(f"Exporting reconstructed volume {recon_path} with shape {shape}")
        for z_index in range(shape[0]):
            image = np.asarray(recon_ds[z_index, :, :], dtype=np.float32)

            float_path = float32_folder / f"slice_{z_index:05d}.tif"
            tifffile.imwrite(float_path, image, photometric="minisblack")

            uint16_path = uint16_folder / f"slice_{z_index:05d}.tif"
            image_u16 = normalise_to_uint16(image, vmin=uint16_vmin, vmax=uint16_vmax)
            tifffile.imwrite(uint16_path, image_u16, photometric="minisblack")

            if z_index % 100 == 0:
                print(f"  exported slice {z_index + 1:,}/{shape[0]:,}")

    print("Reconstructed-volume export complete.")
    print(f"  float32 TIFF folder: {float32_folder}")
    print(f"  uint16 TIFF folder : {uint16_folder}")

In [ ]:
# Reconstructed-volume export.
# Use this path if your HDF already contains a reconstructed 3D volume.
if recon_path is None:
    print("recon_path is None, so reconstructed-volume export is skipped.")
    print("If candidate detection found a reconstructed volume, set recon_path and run this cell again.")
else:
    try:
        export_hdf_recon_dataset_to_tiff_stacks(
            recon_file,
            recon_path,
            FINAL_FLOAT32_FOLDER,
            FINAL_UINT16_FOLDER,
            uint16_vmin=UINT16_VMIN,
            uint16_vmax=UINT16_VMAX,
        )
    except MemoryError:
        print("MemoryError during reconstructed-volume export. This cell reads one slice at a time,")
        print("so check available disk space and whether individual slices are unusually large.")
        raise

In [ ]:
def get_projection_geometry(proj_file, proj_path):
    """Return (n_angles, detector_rows, detector_cols) for the selected projection dataset."""
    proj_file = _normalise_file_path(proj_file, default=INPUT_HDF_PATH)
    with h5py.File(proj_file, "r") as h5:
        proj_ds, proj_path = _require_dataset(h5, proj_path, "projection")
        if proj_ds.ndim != 3:
            raise ValueError(
                f"Projection dataset {proj_file}::{proj_path} has shape {proj_ds.shape}; expected "
                "(n_angles, detector_rows, detector_cols)."
            )
        return tuple(int(value) for value in proj_ds.shape)


# Work out the middle detector-row slab used for preview reconstruction.
projection_shape = get_projection_geometry(proj_file, proj_path)
n_angles, detector_rows, detector_cols = projection_shape

preview_row_count = int(min(PREVIEW_ROW_COUNT, detector_rows))
preview_row_start = max(0, detector_rows // 2 - preview_row_count // 2)
preview_row_stop = min(detector_rows, preview_row_start + preview_row_count)
preview_row_start = max(0, preview_row_stop - preview_row_count)

print(f"Projection shape: n_angles={n_angles}, detector_rows={detector_rows}, detector_cols={detector_cols}")
print(f"Preview slab rows: [{preview_row_start}:{preview_row_stop}] ({preview_row_stop - preview_row_start} rows)")

In [ ]:
# Preview reconstruction of the middle detector slab.
# This is the safest first reconstruction test before running the full scan.

estimated_center = None

try:
    preview_proj, preview_flat, preview_dark, preview_theta = load_projection_slab(
        proj_file,
        proj_path,
        preview_row_start,
        preview_row_stop,
        flat_path=flat_path,
        dark_path=dark_path,
        theta_path=theta_path,
        flat_file=flat_file,
        dark_file=dark_file,
        theta_file=theta_file,
        image_key_file=image_key_file,
        image_key_path=image_key_path,
        image_key_data_file=image_key_data_file,
        image_key_data_path=image_key_data_path,
        flat_image_key_values=FLAT_IMAGE_KEY_VALUES,
        dark_image_key_values=DARK_IMAGE_KEY_VALUES,
        projection_image_key_values=PROJECTION_IMAGE_KEY_VALUES,
    )

    preview_recon, estimated_center = reconstruct_slab(
        preview_proj,
        preview_flat,
        preview_dark,
        preview_theta,
        center=None,
        apply_stripe_removal=True,
        algorithm="gridrec",
    )

    print(f"Estimated centre of rotation: {estimated_center:.3f}")
    print(f"Preview reconstruction shape: {preview_recon.shape}")
    show_middle_slice(preview_recon, title="Preview reconstruction")

    print("Saving preview float32 TIFFs...")
    save_tiff_stack_float32(preview_recon, PREVIEW_FLOAT32_FOLDER)

    print("Saving preview uint16 TIFFs...")
    save_tiff_stack_uint16(
        preview_recon,
        PREVIEW_UINT16_FOLDER,
        vmin=UINT16_VMIN,
        vmax=UINT16_VMAX,
    )

except ImportError as exc:
    print(exc)
except MemoryError:
    print("MemoryError during preview reconstruction.")
    print("Reduce PREVIEW_ROW_COUNT and try again. For example, set PREVIEW_ROW_COUNT = 16.")
    raise

In [ ]:
# Centre testing.
# This saves one middle reconstructed slice per centre value as uint16 TIFF.
# Inspect the outputs, then manually set best_center in the next cell.

CENTER_TEST_OFFSETS = np.arange(-30.0, 31.0, 5.0, dtype=np.float32)

if estimated_center is None:
    print("No estimated center is available yet. Run the preview reconstruction cell first.")
else:
    center_values = [float(estimated_center + offset) for offset in CENTER_TEST_OFFSETS]
    print("Testing centre values:")
    print(", ".join(f"{value:.2f}" for value in center_values))

    for center_value in center_values:
        try:
            test_recon, _ = reconstruct_slab(
                preview_proj,
                preview_flat,
                preview_dark,
                preview_theta,
                center=center_value,
                apply_stripe_removal=True,
                algorithm="gridrec",
            )
            middle_slice = test_recon[test_recon.shape[0] // 2]
            out_path = CENTER_TEST_FOLDER / f"center_{center_value:09.3f}.tif"
            tifffile.imwrite(
                out_path,
                normalise_to_uint16(middle_slice, vmin=UINT16_VMIN, vmax=UINT16_VMAX),
                photometric="minisblack",
            )
            print(f"  saved {out_path.name}")
        except MemoryError:
            print("MemoryError during centre testing. Reduce PREVIEW_ROW_COUNT and try again.")
            raise

    print()
    print(f"Centre-test TIFFs saved to: {CENTER_TEST_FOLDER}")

In [ ]:
# After inspecting the centre-test TIFFs, manually set the best centre here.
# Example:
# best_center = 1234.5

best_center = estimated_center
print(f"best_center = {best_center}")

In [ ]:
def write_full_reconstruction_metadata(
    output_folder,
    proj_file,
    proj_path,
    flat_file,
    flat_path,
    dark_file,
    dark_path,
    theta_file,
    theta_path,
    image_key_file,
    image_key_path,
    image_key_data_file,
    image_key_data_path,
    projection_shape,
    chunk_size,
    center,
):
    lines = [
        "Diamond HDF raw-projection reconstruction TIFF export",
        f"projection_file: {Path(proj_file)}",
        f"projection_path: {proj_path}",
        f"flat_file: {flat_file}",
        f"flat_path: {flat_path}",
        f"dark_file: {dark_file}",
        f"dark_path: {dark_path}",
        f"theta_file: {theta_file}",
        f"theta_path: {theta_path}",
        f"image_key_file: {image_key_file}",
        f"image_key_path: {image_key_path}",
        f"image_key_data_file: {image_key_data_file}",
        f"image_key_data_path: {image_key_data_path}",
        f"projection_shape: {projection_shape}",
        f"chunk_size_detector_rows: {chunk_size}",
        f"centre_of_rotation: {center}",
        "algorithm: tomopy.recon algorithm='gridrec'",
        "stripe_correction: tomopy.remove_stripe_fw",
        "mask: tomopy.circ_mask axis=0 ratio=0.95",
        "",
        "voxel_size_note:",
        "Check the HDF inspection output for pixel/voxel size attributes and set voxel size manually in Dragonfly.",
    ]
    write_metadata_text(output_folder, "reconstruction_metadata.txt", lines)


# Full raw projection reconstruction.
# This cell reconstructs in detector-row chunks and saves each chunk immediately as TIFF slices.
# It will not run unless RUN_FULL_RECONSTRUCTION is True.

if not RUN_FULL_RECONSTRUCTION:
    print("RUN_FULL_RECONSTRUCTION is False, so full reconstruction is skipped.")
    print("After preview and centre testing, set RUN_FULL_RECONSTRUCTION = True in the setup cell and rerun from there.")
else:
    if not TOMOPY_AVAILABLE:
        raise ImportError(_tomopy_missing_message()) from TOMOPY_IMPORT_ERROR

    if best_center is None:
        raise ValueError(
            "best_center is None. Run preview reconstruction and centre testing first, "
            "or manually set best_center."
        )

    projection_shape = get_projection_geometry(proj_file, proj_path)
    n_angles, detector_rows, detector_cols = projection_shape
    center_to_use = float(best_center)

    print("Starting full reconstruction")
    print(f"  projection shape: {projection_shape}")
    print(f"  chunk size: {CHUNK_SIZE} detector rows")
    print(f"  centre: {center_to_use:.3f}")
    print(f"  float32 output: {FINAL_FLOAT32_FOLDER}")
    print(f"  uint16 output : {FINAL_UINT16_FOLDER}")

    write_full_reconstruction_metadata(
        FINAL_FLOAT32_FOLDER,
        proj_file,
        proj_path,
        flat_file,
        flat_path,
        dark_file,
        dark_path,
        theta_file,
        theta_path,
        image_key_file,
        image_key_path,
        image_key_data_file,
        image_key_data_path,
        projection_shape,
        CHUNK_SIZE,
        center_to_use,
    )
    write_full_reconstruction_metadata(
        FINAL_UINT16_FOLDER,
        proj_file,
        proj_path,
        flat_file,
        flat_path,
        dark_file,
        dark_path,
        theta_file,
        theta_path,
        image_key_file,
        image_key_path,
        image_key_data_file,
        image_key_data_path,
        projection_shape,
        CHUNK_SIZE,
        center_to_use,
    )

    for row_start in range(0, detector_rows, int(CHUNK_SIZE)):
        row_stop = min(row_start + int(CHUNK_SIZE), detector_rows)
        print()
        print(f"Reconstructing detector rows [{row_start}:{row_stop}] of {detector_rows}")

        try:
            chunk_proj, chunk_flat, chunk_dark, chunk_theta = load_projection_slab(
                proj_file,
                proj_path,
                row_start,
                row_stop,
                flat_path=flat_path,
                dark_path=dark_path,
                theta_path=theta_path,
                flat_file=flat_file,
                dark_file=dark_file,
                theta_file=theta_file,
                image_key_file=image_key_file,
                image_key_path=image_key_path,
                image_key_data_file=image_key_data_file,
                image_key_data_path=image_key_data_path,
                flat_image_key_values=FLAT_IMAGE_KEY_VALUES,
                dark_image_key_values=DARK_IMAGE_KEY_VALUES,
                projection_image_key_values=PROJECTION_IMAGE_KEY_VALUES,
            )

            recon_chunk, _ = reconstruct_slab(
                chunk_proj,
                chunk_flat,
                chunk_dark,
                chunk_theta,
                center=center_to_use,
                apply_stripe_removal=True,
                algorithm="gridrec",
            )

            print("Saving full-reconstruction float32 TIFF chunk...")
            save_tiff_stack_float32(
                recon_chunk,
                FINAL_FLOAT32_FOLDER,
                start_index=row_start,
                progress_every=100,
            )

            print("Saving full-reconstruction uint16 TIFF chunk...")
            save_tiff_stack_uint16(
                recon_chunk,
                FINAL_UINT16_FOLDER,
                start_index=row_start,
                vmin=UINT16_VMIN,
                vmax=UINT16_VMAX,
                progress_every=100,
            )

            print(f"Finished rows [{row_start}:{row_stop}]")

            del chunk_proj, chunk_flat, chunk_dark, chunk_theta, recon_chunk
            gc.collect()

        except MemoryError:
            print("MemoryError during full reconstruction.")
            print("Reduce CHUNK_SIZE in the setup cell, for example to 32, 16, or 8, then restart the kernel and retry.")
            raise

    print()
    print("Full reconstruction complete.")

In [ ]:
# Verify TIFF output.
# By default this checks the final uint16 folder. Change VERIFY_TIFF_FOLDER if you want to check preview output.

VERIFY_TIFF_FOLDER = FINAL_UINT16_FOLDER


def verify_tiff_output(tiff_folder):
    tiff_folder = Path(tiff_folder)
    tiff_files = sorted(tiff_folder.glob("slice_*.tif"))
    print(f"TIFF folder: {tiff_folder}")
    print(f"TIFF count : {len(tiff_files)}")

    if not tiff_files:
        print("No slice_*.tif files found in this folder.")
        print("Check whether you exported a reconstructed volume, ran preview reconstruction, or enabled full reconstruction.")
        return None

    middle_path = tiff_files[len(tiff_files) // 2]
    image = tifffile.imread(middle_path)
    print(f"Middle TIFF: {middle_path.name}")
    print(f"  shape: {image.shape}")
    print(f"  dtype: {image.dtype}")
    print(f"  min  : {np.nanmin(image)}")
    print(f"  max  : {np.nanmax(image)}")

    plt.figure(figsize=(7, 7))
    plt.imshow(image, cmap="gray")
    plt.title(f"{middle_path.name}")
    plt.axis("off")
    plt.colorbar(shrink=0.75)
    plt.show()
    return image


verified_image = verify_tiff_output(VERIFY_TIFF_FOLDER)

In [ ]:
print("Dragonfly import instructions")
print("-----------------------------")
print(f"1. In Dragonfly, import the uint16 TIFF sequence first: {FINAL_UINT16_FOLDER}")
print(f"2. Keep the float32 TIFF sequence as the scientific copy: {FINAL_FLOAT32_FOLDER}")
print("3. Import as an image sequence/stack using the slice_00000.tif, slice_00001.tif, ... filenames.")
print("4. Check voxel size manually in Dragonfly.")
print("5. Use source_metadata.txt or reconstruction_metadata.txt plus the HDF inspection output to find pixel/voxel size and units.")